# Exercise 1: Kalman Filter for Landmark Localization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(123456)

In this problem, we are going to localize a set of landmarks given knowledge about a robot's motion. First, we can define the robot's motion model (not to be confused with the dynamics model that we will use to formulate the Kalman filter later).

In [ ]:
def robot_motion_model(
    x: np.ndarray, 
    v: float, 
    omega: float,
    dt: float,
) -> np.ndarray:
    """
    Robot motion model.

    Args:
        x: robot state (pose), [px, py, θ]
        v: linear velocity command
        omega: angular velocity command
        dt: timestep to apply command input

    Returns:
        Next robot state (pose)
    """
    return np.array([x[0] + dt * v * np.cos(x[2]),
                     x[1] + dt * v * np.sin(x[2]),
                     x[2] + dt * omega])


def robot_commands(t: float) -> np.ndarray:
    """
    Open-loop robot linear and angular velocity commands.

    Args:
        t: current time
    
    Returns:
        Robot linear and angular velocity commands, shape (2,)
    """
    return np.array([1, np.sin(t)])

# Create array of timesteps
dt = 0.2
tf = 15
time = np.arange(0, tf + dt, dt)

# Ground-truth landmark positions that we will try to estimate.
xm = np.array([
    [0, 0],
    [2, 8],
    [8, 2],
    [10, 10]
], dtype=float)

# Ground-truth robot states.
xr = np.zeros((3, len(time)))
xr[:, 0] = np.array([1, 1, 0])
for i, t in enumerate(time[1:], 1):
    xr[:, i] = robot_motion_model(xr[:, i-1], *robot_commands(t), dt)

### Exercise 1.1 and 1.2: Kalman Filter Implementation
First, assuming we can directly measure the landmark positions, define the matrices $A$ and $C$ that we will use in our Kalman filter implementation. What is the corresponding state the Kalman filter will estimate?

In [ ]:
##### YOUR CODE STARTS HERE #####
# What are the state transition and observation matrices?
raise NotImplementedError("Need to implement code here.")
###### YOUR CODE ENDS HERE ######

# Process and observation noise.
Q = 0.0 * np.eye(8)  # Landmarks are stationary, assume zero process noise.
R = 0.25 * np.eye(8)  # Measurements contain noise, assume observation noise.

def kalman_filter_update(
    prior_mean: np.ndarray, 
    prior_cov: np.ndarray, 
    z: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """
    Given the prior mean and covariance for the set of landmarks, update
    the belief using the given measurements.

    Args:
        prior_mean: prior mean of landmark positions, shape (8,)
        prior_cov: prior covariance matrix, shape (8,8) 
        z: position measurement array, shape (8,)

    Returns:
        Updated mean pose
        Update covariance matrix
    """
    ##### YOUR CODE STARTS HERE #####
    # Implement Kalman filtering predict and update steps.
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######
    return mean, cov 

### Exercise 1.3: Simulate and Analyze Results
Run the code provided below to simulate your Kalman filter with a noisy initial estimate.

In [ ]:
# Ground-truth states (simulated).
x = np.zeros((8, len(time)))
x[:, 0] = xm.flatten()

# Estimated states (Kalman Filter).
mean_kf = np.zeros((8, len(time)))
cov_kf = [np.eye(8) for _ in range(len(time))]

# Initial state and state covariance estimate.
init_mean_std = 0.5
mean_kf[:, 0] = xm.flatten() + init_mean_std * np.random.randn(8)
cov_kf[0] = np.eye(8)
cov_kf[0] = init_mean_std * np.eye(8)

# Run simulation.
for i in range(1, len(time)):
    # Simulate true landmark dynamics
    x[:, i] = A @ x[:, i-1]

    # Simulate noisy received measurement
    z_noise = np.random.multivariate_normal(np.zeros((8,)), R)
    z = C @ x[:, i] + z_noise
    
    # Estimation.
    mean_kf[:, i], cov_kf[i] = kalman_filter_update(mean_kf[:, i-1], cov_kf[i-1], z)

### Plot Mean Estimates of 2D Landmark Positions
Run the provided code below to plot the mean estimates to see how well the filter performed.

In [ ]:
# Plot mean estimates
plt.figure(figsize=(14, 6))

# Subplot for x positions.
plt.subplot(121)
plt.title('Kalman Filter Estimates of Landmark X-coordinates')
for i in range(4):
    true_label = 'Ground Truth' if i == 0 else None
    est_label = 'Estimate' if i == 0 else None
    plt.plot(time, x[2*i, :], linewidth=2, label=true_label, color='r')
    plt.plot(time, mean_kf[2*i, :], '.', markersize=3, label=est_label, color='b')
plt.xlabel('Time')
plt.ylabel('Landmark x-coordinates')
plt.legend()
plt.grid(True)

# Subplot for y positions.
plt.subplot(122)
plt.title('Kalman Filter Estimates of Landmark Y-coordinates')
for i in range(4):
    true_label = 'Ground Truth' if i == 0 else None
    est_label = 'Estimate' if i == 0 else None
    plt.plot(time, x[2*i + 1, :], linewidth=2, label=true_label, color='r')
    plt.plot(time, mean_kf[2*i + 1, :], '.', markersize=3, label=est_label, color='b')
plt.xlabel('Time')
plt.ylabel('Landmark y-coordinates')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Plot Error Ellipses of 2D Landmarks Along Trajectory
Run the provided code below to see a visualization of the uncertainty in the 2D plane.

In [ ]:
def plot_error_ellipse(ax, mean, cov, alpha=1, label=None):
    # Calculate the error ellipse parameters
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
    
    # Compute the radius of the ellipse to correspond to the desired confidence level
    chi2_val = 2.4477  # Corresponds to 95% conf. interval
    width, height = 2 * chi2_val * np.sqrt(eigenvalues)
    
    # Draw the ellipse
    ellipse = patches.Ellipse(mean, width, height, angle=angle, edgecolor='red', fc='None', lw=1, alpha=alpha, label=label)
    ax.add_patch(ellipse)

plt.figure(figsize=(10, 8))
plt.title('Estimated Landmark Positions and Error')
plt.plot(xr[0, :], xr[1, :], linewidth=2, label='Robot Path')
for i in range(4):
    label = 'Landmark' if i == 0 else None
    plt.plot(x[2*i, :], x[2*i+1, :], 'b.', markersize=12, label=label)
for i in range(0, len(time), 10):
    alpha = 0.5 + 0.5 * i / len(time)
    for j in range(4):
        label = 'Uncertainty' if i == 0 and j == 0 else None
        plot_error_ellipse(plt.gca(), mean_kf[2*j:2*(j+1), i], cov_kf[i][2*j:2*(j+1), 2*j:2*(j+1)], alpha=alpha, label=label)
plt.xlabel('X position')
plt.ylabel('Y position')
plt.legend()
plt.grid(True)